# 📄 PDF Question Answering System

An NLP application that allows users to upload a PDF document and ask questions about its content using a Transformer-based Question Answering model.

## Project Overview

This project uses a pre-trained Transformer model to answer questions from PDF documents.

The application extracts text from a PDF, divides it into overlapping chunks, searches all chunks for the most relevant answer, and returns the answer with a confidence score.

## Import Libraries

Import all required libraries for PDF processing, NLP, timing, and data analysis.

In [26]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

## Load Transformer Model

Load a pre-trained Transformer model and tokenizer for extractive Question Answering.

In [27]:
model_name = "deepset/roberta-base-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

print("Model loaded successfully!")

Model loaded successfully!


## Create Question Answering Pipeline

Create a Hugging Face pipeline that combines the tokenizer and model for question answering.

In [28]:
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

print("QA pipeline is ready!")

QA pipeline is ready!


## Upload PDF

Select a PDF document from your computer and extract its text.

In [29]:
root = Tk()
root.withdraw()

pdf_path = filedialog.askopenfilename(
    title="Select a PDF file",
    filetypes=[("PDF files", "*.pdf")]
)

reader = PdfReader(pdf_path)

context = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        context += text + "\n"

print("PDF loaded successfully!")
print("Pages:", len(reader.pages))

PDF loaded successfully!
Pages: 1


## Split PDF into Chunks

Long documents are divided into overlapping chunks to improve answer accuracy.

In [30]:
def split_text(text, chunk_size=900, overlap=150):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = split_text(context)

print("Chunks:", len(chunks))

Chunks: 3


## Ask Questions

Ask unlimited questions about the uploaded PDF. The system searches every chunk and returns the answer with the highest confidence.

In [31]:
from tkinter import Tk, filedialog
from pypdf import PdfReader
import time

def split_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

# Select PDF
root = Tk()
root.withdraw()

pdf_path = filedialog.askopenfilename(
    title="Select a PDF file",
    filetypes=[("PDF files", "*.pdf")]
)

if pdf_path == "":
    print("No file selected.")

else:
    reader = PdfReader(pdf_path)

    context = ""

    for page in reader.pages:
        text = page.extract_text()

        if text:
            context += text + "\n"

    chunks = split_text(context)

    print("PDF loaded successfully!")
    print(f"Pages: {len(reader.pages)}")
    print(f"Chunks: {len(chunks)}")
    print("\nYou can now ask questions about the PDF.")
    print("Type 'exit' to stop.\n")

    qa_history = []

    while True:

        question = input("Ask a question: ").strip()

        if question.lower() == "exit":
            print("\nQuestion Answering ended.")
            break

        if question == "":
            print("Please enter a valid question.")
            continue

        start_time = time.time()

        best_result = None

        for chunk in chunks:

            result = qa_pipeline(
                question=question,
                context=chunk
            )

            if best_result is None or result["score"] > best_result["score"]:
                best_result = result

        inference_time = round(time.time() - start_time, 3)

        confidence = best_result["score"]

        if confidence < 0.2:
            answer = "Answer not found clearly in the PDF."
        else:
            answer = best_result["answer"]

        qa_history.append({
            "Question": question,
            "Answer": answer,
            "Confidence": round(confidence, 4),
            "Inference Time (s)": inference_time
        })

        print("\n" + "=" * 50)
        print(f"Question   : {question}")
        print(f"Answer     : {answer}")
        print(f"Confidence : {confidence:.2%}")
        print(f"Time       : {inference_time} sec")
        print("=" * 50)

PDF loaded successfully!
Pages: 1
Chunks: 3

You can now ask questions about the PDF.
Type 'exit' to stop.


Question   : What's Tala's GPA ??
Answer     : 4.6/5.0
Confidence : 32.00%
Time       : 2.072 sec

Question   : What's her major ?
Answer     : Answer not found clearly in the PDF.
Confidence : 0.58%
Time       : 1.647 sec

Question   : What's Tala's Projects ?
Answer     : Answer not found clearly in the PDF.
Confidence : 18.19%
Time       : 1.74 sec

Question Answering ended.


## Display Results

Display all questions, answers, confidence scores, and inference times in a structured table.

In [32]:
history_df = pd.DataFrame(qa_history)

history_df

,Question,Answer,Confidence,Inference Time (s)
0,What's Tala's GPA ??,4.6/5.0,0.3200,2.072
1,What's her major ?,Answer not found clearly in the PDF.,0.0058,1.647
2,What's Tala's Projects ?,Answer not found clearly in the PDF.,0.1819,1.740


## Conclusion

This project demonstrates how Transformer-based Question Answering models can be used to extract information from PDF documents.

To improve performance on long documents, the PDF is divided into overlapping chunks before inference.

Future work includes implementing Retrieval-Augmented Generation (RAG), semantic search, and a web interface.